# Rust 107: I/O and Files

Chapter 07 of the Rust track, and the one where the language meets everything it does not own: files on disk, bytes on stdout, command-line arguments, environment variables, and other programs. Chapters 01–06 lived inside the process; this chapter crosses the boundary — and the boundary is where Rust's error-as-value philosophy stops being a style choice and becomes the only way to write reliable code.

Everything here is a **runnable example**. Read the markdown, run the cell below it, and change nothing — the practice twin of this notebook is `07_io-and-files_p.ipynb`.

## Who this is written for

You are assumed to be a competent programmer, arriving from Python and C++. You have opened files, parsed `sys.argv`, read environment variables and launched subprocesses before — none of that is new. What is new: every one of those operations returns `Result` in Rust, nothing raises, nothing returns `null`, and the compiler will not let you forget it. The chapter is also the first place the **debug-build** and **platform** facts (Windows, `rustc` 1.98, edition 2024) change what code you are allowed to write — §9 has the details.

## How to read this notebook

- Markdown cells explain; code cells demonstrate. **Run each code cell as you reach it.** Cells are written to be self-sufficient: each one creates what it reads, so a restart-and-run-from-the-top always works.
- Files this chapter creates live under `scratch07/` — the sandbox. Every writing cell makes the directory first, and the last code cell (§12.1) deletes it.
- Code that intentionally *fails* is shown as a comment, together with the diagnostic or panic message you would get. §11 is the catalogue.
- The prose is terse and the facts are dense. This is a reference you will come back to, not a novel you read once.

## The one idea to internalise

**The outside world is fallible, and Rust makes you say so at every step.** A missing file, a closed pipe, a lost environment variable — each comes back as an `Err` you must handle, because there is no exception to catch and no `None` to sneak past the type system. The payoff is a program whose failure map is written in its signatures: `io::Result<()>` tells you every place I/O can go wrong, without reading a line of the body.


### Contents

- [0. Notebook Conventions: How These Cells Run](#0-notebook-conventions-how-these-cells-run)
- [1. The World Outside the Program: Errors as Values](#1-the-world-outside-the-program-errors-as-values)
- [2. stdout and stderr](#2-stdout-and-stderr)
- [3. Paths: `Path`, `PathBuf` and Platform Truth](#3-paths-path-pathbuf-and-platform-truth)
- [4. Reading and Writing Whole Files](#4-reading-and-writing-whole-files)
- [5. File Handles, `OpenOptions` and Append](#5-file-handles-openoptions-and-append)
- [6. Buffered I/O: `BufReader` and `BufWriter`](#6-buffered-io-bufreader-and-bufwriter)
- [7. Working with Directories](#7-working-with-directories)
- [8. Command-Line Arguments](#8-command-line-arguments)
- [9. Environment Variables](#9-environment-variables)
- [10. Running Other Programs](#10-running-other-programs)
- [11. Reading the Compiler: Panics and E-Codes](#11-reading-the-compiler-panics-and-e-codes)
- [12. Performance Notes for Low-Latency Code](#12-performance-notes-for-low-latency-code)
- [13. Mental Model: The Program as a Guest](#13-mental-model-the-program-as-a-guest)
- [14. Habits and Review](#14-habits-and-review)
- [15. Cheat Sheet](#15-cheat-sheet)
- [16. Review Questions](#16-review-questions)

If you are returning to this chapter later, §15 is the one-page summary, §11 is the diagnostics catalogue, and §12 is where the latency notes live.


## 0. Notebook Conventions: How These Cells Run

This notebook runs on the **evcxr** kernel, which wraps each cell into a generated program and compiles it incrementally. Three consequences matter for every cell below:

- **There is no `fn main`.** The kernel supplies one; you are typing its body.
- **State persists across cells.** A `let` from cell 4 is still live in cell 30 — including `use` statements.
- **Items are order-independent, statements are not.** Two `fn`s may call each other regardless of order; two `let`s may not use each other's names before they run.

There is **one namespace per notebook**. Two items with the same name collide exactly as they would in one source file, even in different sections. If you re-run a cell that defines an item, it is redefined; on a redefinition error, restart the kernel and run from the top.

A failing cell does not poison the kernel — the deliberate compile errors in this chapter are comments, so every cell still runs. `println!` writes to stdout; `eprintln!` and `dbg!` write to stderr (§2).

### The chapter sandbox

Every file this chapter touches lives under `scratch07/`, relative to wherever the kernel was started. Each writing cell begins with `fs::create_dir_all("scratch07")`, which succeeds both when the directory is missing and when it already exists — so cells are re-runnable in any order, and the whole sandbox is deleted by the last cell that uses it.

### A note on imports

`use` paths are imported **exactly once**, in the cell that first needs them: `std::fs` in cell 0.1, `std::io` in §1, the `Read`/`Write`/`BufRead` traits where they first apply. In evcxr an import stays in effect for the rest of the session, so later cells rely on earlier ones — after a kernel restart, run from the top and everything is back in scope.


In [ ]:
// 0.1 Smoke test: the sandbox directory exists, and the kernel can see the filesystem.
use std::fs;
fs::create_dir_all("scratch07").unwrap();
println!("cwd contents include the sandbox: {:?}", fs::read_dir(".").unwrap().count() > 0);


## 1. The World Outside the Program: Errors as Values

Every operation in this chapter returns `io::Result<T>` — `Ok(value)` when the world cooperated, `Err(io::Error)` when it did not. There is no exception mechanism to fall back on and no implicit `null`. That single design decision reshapes everything: error handling is code you write on the happy path's shoulder, visible in signatures, checked by the compiler.

| Python | C++ | Rust |
|---|---|---|
| `open()` raises `FileNotFoundError` | `std::ifstream` fails silently until you check `.fail()` | `fs::read_to_string(..)` returns `Err(io::Error)` you **must** handle |

`io::Error` carries a machine-readable `ErrorKind` plus a human-readable message. You saw `Result` accessors in chapter 01; here they meet real I/O, and chapter 11 will give the type design its full treatment.

### The first contact: a missing file


In [ ]:
// 1.1 Reading a file that does not exist yields an Err you can classify by kind.
// std::fs arrived in cell 0.1; std::io starts here.
use std::io;
let read = fs::read_to_string("scratch07/config.toml");
match read {
    Ok(text) => println!("loaded config: {text:?}"),
    Err(e) if e.kind() == io::ErrorKind::NotFound => {
        println!("no config yet ({e}); starting from defaults");
    }
    Err(e) => println!("unexpected I/O problem: {e}"),
}


Cell 1.1 matched on `e.kind()`, the machine-readable classification — `NotFound`, `PermissionDenied`, `InvalidData` and friends. Matching kinds is how a program *decides* what to do; the `Display` form of the error (`{e}`) is how it *reports* what happened. The two serve different masters: logic consumes kinds, humans read messages.

### Classifying two real failures


In [ ]:
// 1.2 Two real errors, two kinds: a missing file, and a directory opened as a file.
fs::create_dir_all("scratch07/subdir").unwrap();
for target in ["scratch07/absent.txt", "scratch07/subdir"] {
    match fs::File::open(target) {
        Ok(_) => println!("{target}: opened"),
        Err(e) => println!("{target}: kind={:?} message={e}", e.kind()),
    }
}
// kind=NotFound message=The system cannot find the file specified. (os error 2)
// kind=PermissionDenied message=Access is denied. (os error 5)   <- Windows


The second error is a platform lesson in disguise: opening a directory as a file is `PermissionDenied` on Windows (the message is the OS's own, `os error 5`). The `kind` is the portable part you can program against; the message text is whatever the operating system felt like saying.

### Turning `Err` into a domain error

`map_err` converts the I/O error into whatever type your function's callers should see — here a plain `String`, which is enough until chapter 11 introduces real error types.


In [ ]:
// 1.3 map_err converts the OS error into a domain error with context.
/// Load a config file, reporting failures as a self-contained message.
fn load_config(path: &str) -> Result<String, String> {
    fs::read_to_string(path)
        .map_err(|e| format!("{path}: could not load config ({e})"))
}
fs::create_dir_all("scratch07").unwrap();
fs::write("scratch07/config.toml", "threads = 4\n").unwrap();
println!("good  -> {:?}", load_config("scratch07/config.toml"));
println!("bad   -> {:?}", load_config("scratch07/missing.toml"));


## 2. stdout and stderr

Two output streams, two audiences: **stdout** carries the program's data — the lines other programs and files consume; **stderr** carries diagnostics — what the *human running* the program needs to see. They are separate channels precisely so the data can be redirected without dragging the chatter along: `myprog > out.txt` captures stdout only.

| Macro | Stream | Buffered | Use for |
|---|---|---|---|
| `print!` / `println!` | stdout | line-buffered (terminal), block-buffered (pipe) | data |
| `eprint!` / `eprintln!` | stderr | unbuffered | diagnostics |
| `dbg!` | stderr | unbuffered | quick debug dumps |

The buffering column is a low-latency fact: stderr arrives immediately because it is unbuffered, which is exactly what you want from a diagnostic channel — §12 returns to the cost side.

### Seeing the two streams


In [ ]:
// 2.1 Data on stdout, diagnostics on stderr — two channels from one cell.
let price = 4125.5;
println!("price={price}");                 // stdout: the data
eprintln!("recomputed price at tick end"); // stderr: the narration
// In Jupyter both appear below the cell. In a terminal:
//   program > data.txt   captures stdout only; stderr still reaches the screen.


### `dbg!`: the diagnostic that returns its input

`dbg!` prints an expression with its file, line and value to stderr **and returns the value**, so it can wrap an expression without disturbing the surrounding code. It is the one-liner replacement for a temporary `println!`.


In [ ]:
// 2.2 dbg! prints file:line and value to stderr, and hands the value back.
let raw = 4125.5f64;
let rounded = dbg!(raw * 1.08f64).round();
println!("rounded = {rounded}"); // rounded is a normal f64; dbg! was just a window


### Who is listening? `is_terminal`

`stdout().is_terminal()` reports whether stdout is an interactive terminal or something captured (a file, a pipe — or a notebook). Many programs use it to decide whether colour codes are welcome.


In [ ]:
// 2.3 is_terminal tells captured output from a real terminal.
// the io module itself is already in scope from §1
use std::io::IsTerminal;
println!("stdout is a terminal: {}", io::stdout().is_terminal());
// Run this same program from a shell and the answer flips to true;
// under Jupyter the output is captured, so it usually reports false.


## 3. Paths: `Path`, `PathBuf` and Platform Truth

A **path** is not a string — that is the first platform lesson. Windows uses `\` separators and drive letters, Unix uses `/`; path *text* is platform-shaped, but path *types* are not. Rust splits the job in two:

| Type | Analogy | Mutability | Use |
|---|---|---|---|
| `&Path` | like `&str` — a borrowed view | read-only | function parameters |
| `PathBuf` | like `String` — an owned buffer | growable | building paths, storing them |

Almost every `std::fs` function takes `impl AsRef<Path>`, so `&str` literals, `&Path` and `&PathBuf` all work at call sites without conversion. Keep that rule: **functions take `&Path` (or anything `AsRef<Path>`), programs build `PathBuf`**.

### The component view


In [ ]:
// 3.1 A path is a sequence of components, not a string to split on '/'.
use std::path::{Path, PathBuf};
let p = Path::new("logs/app/2026-09-25.log");
println!("parent     = {:?}", p.parent());
println!("file_name  = {:?}", p.file_name());
println!("stem       = {:?}", Path::new("archive.tar.gz").file_stem());
println!("extension  = {:?}", Path::new("archive.tar.gz").extension());
// `file_stem` strips only the *last* extension: "archive.tar", not "archive".


`file_stem` on `archive.tar.gz` gives `archive.tar` — only the final extension is stripped. That surprises people coming from `Pathlib(...).stem`, which would give `archive`.

### Building paths: `join`, `push`, `set_extension`

`join` and `push` append a component; `set_extension` replaces the suffix. One Windows fact you must not learn the hard way: **`push` and `join` normalise separators to `\` on Windows** — you get backslashes out even when you pushed components joined by `/`. If you need a forward-slash string (URLs, config keys), convert explicitly.


In [ ]:
// 3.2 join and push build paths; set_extension rewrites the suffix.
let mut log_path = PathBuf::from("scratch07").join("run");
log_path.set_extension("log");
println!("built: {log_path:?}");
let replaced = PathBuf::from("relative").join("C:/abs");
println!("joining an absolute path replaces the base: {replaced:?}");
// On Windows `log_path` prints with a backslash: "scratch07\\run.log".


The `join` column has one more trap from the table above: joining an *absolute* path (`C:/abs`) **replaces** the base entirely — handy for overriding, nasty when accidental.

### The full-circle functions

Three functions answer "where is this, really, and does it exist": `canonicalize` returns the absolute, OS-canonical form (on Windows, prefixed with the `\\?\` verbatim marker); `fs::exists` answers the existence question without erroring on a missing file (stabilised in Rust 1.83 — safe to use on this toolchain); `current_dir` and `set_current_dir` anchor the process.


In [ ]:
// 3.3 canonicalize, exists and current_dir round out the toolkit.
fs::create_dir_all("scratch07").unwrap();
fs::write("scratch07/anchor.txt", b"x").unwrap();
let canon = fs::canonicalize("scratch07/anchor.txt").unwrap();
println!("canonical form: {canon:?}"); // "\\?\C:\...\scratch07\anchor.txt" on Windows
println!("exists (yes/no): {} / {}", fs::exists("scratch07/anchor.txt").unwrap(),
                                      fs::exists("scratch07/ghost.txt").unwrap());
println!("cwd is: {:?}", std::env::current_dir().unwrap().file_name());
let _ = PathBuf::new(); let _ = Path::new(".");


## 4. Reading and Writing Whole Files

For the common case — a file you want *all* of, or a payload you want written *now* — `std::fs` has three one-shot pairs. They allocate the whole result (or take the whole input), which makes them the right choice for config files, small data files and tests, and the wrong choice for anything measured in gigabytes (§6 covers streaming).

| Function | Direction | Input/Output |
|---|---|---|
| `fs::read` | read all | `Vec<u8>` — bytes |
| `fs::read_to_string` | read all | `String` — errors on invalid UTF-8 |
| `fs::write` | write all | takes anything byte-like, **truncates** |

`fs::write` truncates: an existing file's content is gone before your new bytes land. For "add to what is there", §5's append mode is the tool.

### The round trip


In [ ]:
// 4.1 write, read, read_to_string: the whole-file round trip.
fs::create_dir_all("scratch07").unwrap();
fs::write("scratch07/positions.csv", "sym,qty\nAAPL,300\nMSFT,120\n").unwrap();
let text = fs::read_to_string("scratch07/positions.csv").unwrap();
let bytes = fs::read("scratch07/positions.csv").unwrap();
println!("text = {text:?}");
println!("byte count = {} (u8 view of the same file)", bytes.len());
assert_eq!(text.len(), bytes.len());


### Not everything on disk is text

`read_to_string` returns an `InvalidData` error when the bytes are not UTF-8 — there is no lossy mode and no defaulting. The byte-level `fs::read` never has that problem, because it promises nothing about encoding.

### When the file lies about being text


In [ ]:
// 4.2 read_to_string refuses non-UTF-8 bytes; fs::read accepts them.
fs::write("scratch07/utf16.bin", [0xFF, 0xFE, 0x68, 0x00]).unwrap();
match fs::read_to_string("scratch07/utf16.bin") {
    Ok(t) => println!("text: {t:?}"),
    Err(e) => println!("error: kind={:?} message={e}", e.kind()),
    // kind=InvalidData message=stream did not contain valid UTF-8
}
let bytes = fs::read("scratch07/utf16.bin").unwrap();
println!("as bytes, no complaint: {bytes:?}");


That `InvalidData` kind is worth matching deliberately when you read config files: "the file exists but is not the encoding you assumed" deserves a different message from "the file is missing" (§1).

### The atomic-ish move: `rename`

`fs::rename` moves a file. On one filesystem it is an O(1) metadata operation — which is why the "write to temp, rename into place" pattern is the standard way to replace a file without leaving a torn copy if the process dies halfway. (Across filesystem boundaries rename fails; copy-then-delete is the fallback.)


In [ ]:
// 4.3 rename: write to a temp name, then move into place.
fs::create_dir_all("scratch07").unwrap();
fs::write("scratch07/snap.tmp", "day close snapshot").unwrap();
fs::rename("scratch07/snap.tmp", "scratch07/snap.csv").unwrap();
println!("snap.csv now holds: {:?}", fs::read_to_string("scratch07/snap.csv").unwrap());
println!("snap.tmp still exists: {}", fs::exists("scratch07/snap.tmp").unwrap());


## 5. File Handles, `OpenOptions` and Append

`File` is a **handle**: an owned value that holds an OS resource and closes it when dropped — RAII, the pattern chapter 05 promised would follow you everywhere. A dropped `File` releases the descriptor; there is no explicit `close()` to forget.

`File::open` = read-only, `File::create` = create-or-**truncate**, and everything else goes through `OpenOptions`, the builder for exactly which access you want. Note `create(true)` is *not* "only create" — it is "create if missing, open otherwise"; the "fail if it exists" flag is `create_new(true)`.

### The four shapes


In [ ]:
// 5.1 The four ways to get a File handle.
// adds File and OpenOptions next to the fs module already in scope
use std::fs::{File, OpenOptions};
fs::create_dir_all("scratch07").unwrap();
let _ro = File::open("scratch07/positions.csv").unwrap();      // read-only; must exist
let _trunc = File::create("scratch07/trunc.txt").unwrap();     // create or truncate
fs::write("scratch07/trunc.txt", "old content gone").unwrap();
let mut rw = OpenOptions::new().read(true).write(true).open("scratch07/trunc.txt").unwrap();
let mut head = String::new();
use std::io::Read;
rw.read_to_string(&mut head).unwrap();
println!("read back through the rw handle: {head:?}");


### Append mode: logs, ledgers, anything that only grows

Append mode positions every write at the end, no matter where other writes left the cursor — and critically, **the position survives close-and-reopen**. This is the mode every log file and every trade ledger uses, and the reason `OpenOptions` exists instead of a bigger pile of `File::open*` variants.


In [ ]:
// 5.2 Append mode: two sessions, one file, order preserved.
// the writer trait, first needed here
use std::io::Write;
fs::create_dir_all("scratch07").unwrap();
let mut log = OpenOptions::new().append(true).create(true).open("scratch07/trades.log").unwrap();
writeln!(log, "session 1: buy 100 AAPL @ 4125.5").unwrap();
drop(log); // session 1 ends; the handle is closed by the drop
let mut log = OpenOptions::new().append(true).open("scratch07/trades.log").unwrap();
writeln!(log, "session 2: sell 100 AAPL @ 4131.0").unwrap();
println!("--- file after two sessions ---");
println!("{}", fs::read_to_string("scratch07/trades.log").unwrap());


Cell 5.2 redeclares `log` in the same cell — shadowing, not redefinition, because they are `let` bindings, not items. Two handles at once on the *same* file is legal; two live `&mut` borrows of one handle are not (§11, E0499).

### Write a formatted line: the `write!` macros

Handles implement `std::io::Write`, so `write!`/`writeln!` format into them exactly like `format!` builds a `String`. The difference that bites: **`write!` returns the `io::Result`** — you unwrap it like any other I/O result, and the macro returns `()` in the success case, not the formatted text.


In [ ]:
// 5.3 write!/writeln! format straight into the handle; the Result must be handled.
let mut f = File::create("scratch07/quotes.txt").unwrap();
write!(f, "MSFT ").unwrap();          // same success value (), text kept on one line
writeln!(f, "{}", 231.75).unwrap();   // writeln! adds the newline, then flushes the line
writeln!(f, "AAPL {}", 4125.5).unwrap();
println!("{}", fs::read_to_string("scratch07/quotes.txt").unwrap());


## 6. Buffered I/O: `BufReader` and `BufWriter`

Every `read`/`write` call on a `File` becomes a **system call**, and system calls cost microseconds while memory costs nanoseconds. `BufReader`/`BufWriter` put an in-memory buffer between your code and the OS: readers fetch big chunks and serve small requests from RAM; writers collect small writes and hand the OS big ones.

| Type | Wraps | Wins by |
|---|---|---|
| `BufReader<R>` | any `Read` | fewer, larger read syscalls; `read_line`/`lines` on top |
| `BufWriter<W>` | any `Write` | fewer, larger write syscalls |
| `BufWriter::with_capacity(n, w)` | — | your buffer size instead of the 8 KiB default |

### The rule that saves your data

A `BufWriter` holds back whatever has not filled its buffer. The data leaves when: the buffer fills, **you call `.flush()`**, or the writer is **dropped** (dropping flushes — but the `Drop` handler *cannot report* an error: if the final flush fails, the data is gone silently. Batch work ends with an explicit `flush().unwrap()`.)

### Watching small writes sit in memory


In [ ]:
// 6.1 Two tiny writes never reach the file until flush is called.
// the buffered writer; the Write trait is already in scope from §5.2
use std::io::BufWriter;
fs::create_dir_all("scratch07").unwrap();
let file = File::create("scratch07/buf.txt").unwrap();
let mut w = BufWriter::with_capacity(8, file);
write!(w, "ab").unwrap();
write!(w, "cd").unwrap();
let before = fs::read_to_string("scratch07/buf.txt").unwrap();
println!("before flush: {before:?} (both writes are still in memory)");
w.flush().unwrap();
let after = fs::read_to_string("scratch07/buf.txt").unwrap();
println!("after flush: {after:?}");


The probed behaviour, worth knowing in both directions: a single `write!` **larger than the buffer capacity bypasses the buffer entirely** and goes straight to the file — buffering only helps writes that fit.

### Streaming lines from a reader

On the read side, `BufReader` adds line access: `read_line` for one line (keeping the `\n`), `lines()` for an iterator that strips it. `lines()` yields `io::Result<String>` per item — reading a directory feed is now a `for` loop with `?` or `unwrap` in the body.


In [ ]:
// 6.2 read_line keeps the newline; lines() strips it.
// the reader side of the same trait family
use std::io::{BufRead, BufReader};
fs::create_dir_all("scratch07").unwrap();
fs::write("scratch07/feed.txt", "AAPL 4125.5\nMSFT 231.75\nNVDA 187.2").unwrap();
let f = File::open("scratch07/feed.txt").unwrap();
let mut r = BufReader::new(f);
let mut line = String::new();
let n = r.read_line(&mut line).unwrap();
println!("read_line: {n} bytes, {line:?}");   // keeps "\n"
let lines: Vec<String> = r.lines().map(|l| l.unwrap()).collect();
println!("lines(): {lines:?}");               // newlines stripped


Cell 6.2's file ends **without** a trailing newline, and `lines()` still yields the final partial line — no sentinel handling for you to write. Note also what `read_line` appends into: the `String` you pass is **extended, not replaced** — clear it (or use a fresh one) between lines.

### A drop-in File replacement for examples: reading from memory

`&[u8]` implements `Read` and `BufRead` directly (`io::Cursor` wraps an owned `Vec<u8>` and adds `Seek`). Any function written against `impl BufRead` runs unchanged against files, sockets — or bytes in a test.

### The same line-reading code with no file at all


In [ ]:
// 6.3 &[u8] implements BufRead: identical line code, zero filesystem.
let data: &[u8] = b"one\ntwo\nthree\n";
let mut r = data;
let mut buf = String::new();
r.read_line(&mut buf).unwrap();
println!("first line: {buf:?}");
let rest: Vec<String> = r.lines().map(|l| l.unwrap()).collect();
println!("rest: {rest:?}");


## 7. Working with Directories

Directories get one-shot constructors (`create_dir`, `create_dir_all`, `remove_dir`, `remove_dir_all`) plus **`read_dir`** — the directory iterator. `read_dir` returns each entry as a `Result`, and each entry's fields are again fallible; the `.filter_map(|e| e.ok())` idiom (or `.map(|e| e.unwrap())` when a failure should stop everything) is how you meet it. **Read order is OS-dependent — always sort** before showing anything.

### Listing a directory


In [ ]:
// 7.1 read_dir iterates entries; sort before showing.
fs::create_dir_all("scratch07/quotes").unwrap();
fs::write("scratch07/quotes/aapl.csv", "").unwrap();
fs::write("scratch07/quotes/msft.csv", "").unwrap();
fs::write("scratch07/quotes/readme.md", "").unwrap();
let mut names: Vec<String> = fs::read_dir("scratch07/quotes")
    .unwrap()
    .map(|e| e.unwrap().file_name().to_string_lossy().into_owned())
    .collect();
names.sort();
println!("sorted entries: {names:?}");


`file_name()` hands back an `OsStr` — a filename is *bytes the OS accepts*, not guaranteed UTF-8. `.to_string_lossy()` is the pragmatic bridge; the honest alternative is keeping `PathBuf`s end to end and never converting.

### From names to metadata

Each `DirEntry` carries `metadata()` — size, type, timestamps. `is_file`/`is_dir` save you opening anything; `len` is bytes on disk.

### Metadata without opening the file


In [ ]:
// 7.2 DirEntry::metadata gives type and size without opening the file.
fs::create_dir_all("scratch07/archive").unwrap();
fs::write("scratch07/quotes/aapl.csv", "close\n4125.5\n").unwrap();
let mut rows: Vec<String> = fs::read_dir("scratch07")
    .unwrap()
    .map(|e| {
        let entry = e.unwrap();
        let md = entry.metadata().unwrap();
        let kind = if md.is_dir() { "dir " } else { "file" };
        format!("{:>5} {kind} {}", md.len(), entry.file_name().to_string_lossy())
    })
    .collect();
rows.sort();
for row in rows { println!("{row}"); }


### Recursion is yours to write

`std::fs` has no built-in recursive walker; `read_dir` plus recursion is the pattern (or the `walkdir`/`jwalk` crates when a dependency is allowed). Building one also exercises the two failure strategies: propagate (`?` — a failure stops the walk) or skip (`filter_map`/`if let` — best effort).

### A recursive size tally — errors propagate


In [ ]:
// 7.3 A recursive walk returning io::Result: the first unreadable entry aborts.
/// Total byte size of a directory tree; the first unreadable entry aborts the walk.
fn tree_size(dir: &Path) -> io::Result<u64> {
    let mut total = 0u64;
    for entry in fs::read_dir(dir)? {
        let entry = entry?;
        let md = entry.metadata()?;
        if md.is_dir() {
            total += tree_size(&entry.path())?;
        } else {
            total += md.len();
        }
    }
    Ok(total)
}
fs::create_dir_all("scratch07/tree/deep/deeper").unwrap();
fs::write("scratch07/tree/a.txt", [0u8; 300]).unwrap();
fs::write("scratch07/tree/deep/b.txt", [0u8; 500]).unwrap();
fs::write("scratch07/tree/deep/deeper/c.txt", [0u8; 200]).unwrap();
println!("tree size = {} bytes", tree_size(Path::new("scratch07/tree")).unwrap());


Cell 7.3 is the chapter's `?` showcase: three fallible operations per entry (`read_dir`, the entry itself, its metadata), all flowing out through one `io::Result` return type — the same signature style §1.3 promised. The recursive call is `tree_size(&entry.path())?` because recursion does not exempt you from the contract.

The lenient variant flips the strategy: unreadable entries are skipped, and the function returns a plain number. Best-effort scans (disk usage dashboards) want this; loaders of critical config want §7.3's.

### The same walk, skip-and-continue


In [ ]:
// 7.4 The lenient walk: unreadable entries are skipped, not fatal.
/// Best-effort byte size; unreadable subtrees contribute zero.
fn tree_size_lenient(dir: &Path) -> u64 {
    let mut total = 0u64;
    if let Ok(entries) = fs::read_dir(dir) {
        for entry in entries.filter_map(|e| e.ok()) {
            if let Ok(md) = entry.metadata() {
                if md.is_dir() {
                    total += tree_size_lenient(&entry.path());
                } else {
                    total += md.len();
                }
            }
        }
    }
    total
}
println!("lenient tally = {} bytes", tree_size_lenient(Path::new("scratch07/tree")));
println!("missing dir tally = {} bytes (no error, just 0)", tree_size_lenient(Path::new("scratch07/ghost")));


## 8. Command-Line Arguments

`std::env::args()` yields the arguments as `String`s, with the **program name as element 0** — the same shape as Python's `sys.argv`, one iterator instead of a list. The two workhorse methods: `.nth(n)` (0-indexed, `Option<String>` — absent arguments are `None`, not an index error) and `.skip(1)` (everything after the program name).

The `OsString` caveat from §7 applies here too: arguments are bytes the OS passed, not guaranteed UTF-8, and `args()` **panics on non-UTF-8 input**. `args_os()` never panics and is what careful CLI libraries build on; for teaching and tooling, `args()` is the everyday choice.

### Reading the argument vector


In [ ]:
// 8.1 args(): element 0 is the program name; nth() is a safe index.
use std::env;
let mut all = env::args();
let program = all.next().unwrap();
let rest: Vec<String> = all.collect();
println!("program = {program:?}");
println!("rest    = {rest:?}");
let third = env::args().nth(2);
println!("third argument (may be None): {third:?}");


Cell 8.1 runs under the kernel, where the "program" is the kernel host and the argument list is nearly empty — the shape is what matters, and the identical code prints real arguments when the notebook's functions run inside a shipped binary. Note what `nth(2)` returned: `Option<String>`, because "no third argument" is a *value*, not a crash — the same philosophy §1 applied to missing files.

### `args_os` for the honest view

`args_os()` yields `OsString`s: no panic on odd bytes, `.len()` available without collecting, and `.to_string_lossy()` when a display string is needed. One count, no panics:

### Counting arguments without assuming UTF-8


In [ ]:
// 8.2 args_os(): the panic-free view of the same vector.
let argv: Vec<_> = env::args_os().collect();
println!("argument count (including program name): {}", argv.len());


## 9. Environment Variables

`std::env::var` returns `Result<String, VarError>` — reading the environment is an I/O-shaped operation like everything else in this chapter: the variable may be absent (`VarError::NotPresent`) or present but non-UTF-8 (`VarError::NotUnicode`). Match on the `VarError` instead of `unwrap`-ing, and a missing variable becomes a branch, not a crash.

Rust reads environment variables **once, eagerly, at the call** — there is no per-access snapshot and no caching to worry about inside one process. The set this program sees is inherited from its parent process (for notebooks: the **Jupyter server's** environment, not your shell's — chapter 09 depends on this fact for `RUST_LOG`).

### Reading: match, don't unwrap


In [ ]:
// 9.1 var() is Result-shaped: absence is a value you can match on.
match env::var("RUST_LOG") {
    Ok(level) => println!("RUST_LOG = {level}"),
    Err(env::VarError::NotPresent) => println!("RUST_LOG is not set; defaults apply"),
    Err(env::VarError::NotUnicode(_)) => println!("RUST_LOG is set to non-UTF-8 bytes"),
}
let path = env::var("PATH").unwrap_or_else(|_| String::new());
println!("PATH present: {} ({} chars)", !path.is_empty(), path.len());


### Writing: an `unsafe` block, and why

Setting or removing a variable — `env::set_var` / `env::remove_var` — is **`unsafe` as of edition 2024**: other threads may be reading the environment concurrently, and the C runtime underneath gives no synchronisation guarantee. In a single-threaded notebook cell the call is safe, so the block is where the program states that claim:

### set_var, remove_var — and which program is running


In [ ]:
// 9.2 set_var/remove_var are unsafe in edition 2024; current_exe names this program.
unsafe { env::set_var("NB_RATE_LIMIT", "40") };
println!("just set: NB_RATE_LIMIT = {:?}", env::var("NB_RATE_LIMIT"));
unsafe { env::remove_var("NB_RATE_LIMIT") };
println!("after remove: {:?}", env::var("NB_RATE_LIMIT"));
println!("this program is: {:?}", env::current_exe().unwrap().file_name());


`current_exe()` names the *host process* — under the kernel, not your notebook. The cell's real content is the `unsafe` blocks: in edition 2021 and earlier the same calls were safe functions; the change is a deliberate signal that mutating a process-global while others read it is a promise the compiler cannot check. Chapter 16 (Concurrency) returns to exactly this class of problem.

## 10. Running Other Programs

`std::process::Command` spawns a child process. Two entry points, two contracts:

| Method | Returns | When to use |
|---|---|---|
| `.output()` | `Result<Output>`: stdout, stderr, status — child runs to completion first | you want the data |
| `.status()` | `Result<ExitStatus>`: exit code only, output goes to your stdout/stderr | you want the side effect |

Output comes back as raw bytes (`Vec<u8>`) — the child chooses its encoding; `String::from_utf8_lossy` is the pragmatic decode. The failure modes split cleanly: `.output()` itself errs when the program **does not exist** (`NotFound`), while a program that *runs but fails* is still an `Ok` output with `status.success() == false` — the exit code is data, not an exception.

### Capturing output from a real program


In [ ]:
// 10.1 output(): run git, capture stdout, decode lossily.
// the process API, first needed here
use std::process::Command;
let out = Command::new("git").arg("--version").output().unwrap();
println!("success    : {}", out.status.success());
println!("stdout     : {}", String::from_utf8_lossy(&out.stdout).trim());
println!("stderr empty: {}", out.stderr.is_empty());


Cell 10.1 assumes `git` is on `PATH` — on this machine it is (git 2.49). If it were not, `output()` would return the same `NotFound` kind as §1's missing file: **a missing program and a missing file are the same error class**, which is exactly why the kinds exist.

### Exit codes are data

A program that starts but fails gives `Ok(...)` with a non-zero code. `.status()` is the variant for side-effecting commands; here it also shows a deliberate non-zero exit, produced with the shell (`cmd /C` on Windows — the Unix equivalent is `sh -c`):

### Running, failing, and reading the code


In [ ]:
// 10.2 status(): the exit code is data; a missing program is the Err case.
let st = Command::new("cmd").args(["/C", "exit 3"]).status().unwrap(); // Unix: sh -c "exit 3"
println!("success: {}, exit code: {:?}", st.success(), st.code());
assert_eq!(st.code(), Some(3));
let missing = Command::new("definitely-not-a-program-07").output();
println!("missing program: {:?}", missing.err().map(|e| e.kind()));


## 11. Reading the Compiler: Panics and E-Codes

I/O code meets four compiler errors and one runtime behaviour often enough that they deserve their own catalogue — every wording below was captured from this exact toolchain.

| Code | Trigger line | Message (abridged) | Fix |
|---|---|---|---|
| **E0277** | `println!("{}", fs::read_to_string(p))` | `` `Result<String, std::io::Error>` doesn't implement `std::fmt::Display` `` | unwrap/match first; a `Result` is not a `String` |
| **E0308** | `let a: Option<&str> = env::args().nth(1)` | ``expected `Option<&str>`, found `Option<String>`` ` with help: *try using `.as_deref()`* | `.nth(1).as_deref()` |
| **E0499** | two `writeln!`s through `&mut f1, &mut f2` of one file | ``cannot borrow `log` as mutable more than once at a time`` | one live `&mut` writer; share sequentially |
| **E0599** | `reader.read_line(..)` without the import | ``no method named `read_line` found`` … *trait `BufRead` which provides `read_line` is implemented but not in scope* | `use std::io::BufRead;` |

E0599 is the trait-bound lesson in miniature: the method exists, the type implements it, but **trait methods need the trait in scope** — the compiler names the missing `use` for you.

### Panics: the failure channel you should not use for I/O

`unwrap`/`expect` on an I/O result converts a recoverable `Err` into a panic. The panic payload shows how the two methods differ: `unwrap` prints the error's `Debug` form; `expect("msg")` prefixes it with your message — still `Debug` for the error itself.

### Catching a panic to read its payload


In [ ]:
// 11.1 Panics are catchable; expect's payload = message + Debug of the error.
let outcome = std::panic::catch_unwind(|| {
    std::fs::read_to_string("scratch07/absent-config.toml").expect("config load failed");
});
match outcome {
    Ok(text) => println!("unexpectedly read: {text:?}"),
    Err(payload) => {
        let msg = payload.downcast_ref::<String>().cloned()
            .or_else(|| payload.downcast_ref::<&str>().map(|s| s.to_string()))
            .unwrap_or_default();
        println!("caught: {msg}");
        // "caught: config load failed: Os { code: 2, kind: NotFound, ... }"
    }
}


`catch_unwind` exists for boundaries — FFI, thread pools, servers that must survive a request — not for control flow. The idiomatic shape stays §1's: `Result` in, `match` out, panic only for the bugs a human must fix. Note the payload's `Os { code: 2, ... }` — the `Debug` form exposing the raw OS error code that `Display` would have hidden.


## 12. Performance Notes for Low-Latency Code

The whole chapter's cost model is one sentence: **every unbuffered `read`/`write` on a `File` is a system call, and system calls are the expensive part** — context switches into the kernel, buffer copies, scheduler wake-ups. Buffering trades memory for syscall count, and the ratio is not subtle. The same logic explains §2's table: stderr is unbuffered so diagnostics arrive immediately, and that immediacy is paid per `eprintln!`.

| Habit | Cost shape | Fix |
|---|---|---|
| `writeln!` in a loop on a bare `File` | one syscall per line | wrap in `BufWriter` |
| `read_to_string` on a multi-GB file | whole file in RAM | `BufReader` + `lines()` streaming |
| `fs::write` per record (append) | open + write + close, each time | keep one append handle open |
| `eprintln!` in a hot loop | unbuffered syscall each call | aggregate, or gate behind a level |

### Measuring it


In [ ]:
// 12.1 The same 3000 lines, unbuffered vs buffered: syscall count is the difference.
// the last new import of the chapter
use std::time::Instant;
fs::create_dir_all("scratch07").unwrap();
let t0 = Instant::now();
{
    let mut f = File::create("scratch07/timing-unbuf.log").unwrap();
    for i in 0..3000u32 {
        writeln!(f, "tick-{i:04} price={:.2}", 4100.0 + i as f64).unwrap();
    }
} // dropped here: 3000 separate syscalls already made
let unbuffered = t0.elapsed();
let t0 = Instant::now();
{
    let file = File::create("scratch07/timing-bufd.log").unwrap();
    let mut w = BufWriter::new(file);
    for i in 0..3000u32 {
        writeln!(w, "tick-{i:04} price={:.2}", 4100.0 + i as f64).unwrap();
    }
    w.flush().unwrap(); // the one syscall the buffer was waiting for
}
let buffered = t0.elapsed();
println!("unbuffered: {unbuffered:?}");
println!("buffered  : {buffered:?}");
assert_eq!(fs::read("scratch07/timing-unbuf.log").unwrap(),
           fs::read("scratch07/timing-bufd.log").unwrap());
println!("identical bytes, wildly different cost");


On this machine the gap is tens of milliseconds versus **under a millisecond** — roughly **30–50×** depending on the run — and it widens as records get smaller and counts grow. The block-scoped `{ .. }` around each writer makes the drop point explicit: the unbuffered timing includes nothing but its syscalls, and the buffered one ends with the `flush()` the §6 rules require.

### Chapter cleanup


In [ ]:
// 12.2 The sandbox goes away; every cell above can rebuild it from nothing.
fs::remove_dir_all("scratch07").unwrap();
println!("sandbox removed: {}", !fs::exists("scratch07").unwrap());


## 13. Mental Model: The Program as a Guest

Your process did not create the filesystem, the environment, or the other programs it talks to — it is a **guest**, and every host interaction can fail or answer differently than expected. The chapter's types are the guest's manners:

| Host thing | Rust type | The failure it forces you to name |
|---|---|---|
| files | `fs::*` → `io::Result<T>` | missing (`NotFound`), unreadable (`PermissionDenied`), wrong encoding (`InvalidData`) |
| directories | `read_dir` → iterator of `Result`s | unreadable entries, mid-walk races |
| arguments | `args()` / `args_os()` | absent argument = `None`; non-UTF-8 = panic or `OsString` |
| environment | `var` → `Result<String, VarError>` | `NotPresent`, `NotUnicode` |
| other programs | `Command` → `Result<Output>` | `NotFound` to start; exit code as data once running |
| your own streams | `stdout`/`stderr` handles | pipes can close; writes are `Result`s too |

One `Result`-returning function can now combine several of these — and because every failure is a value in the same channel, `?` composes them without a single `try/except` in sight:

### The chapter in one function


In [ ]:
// 13.1 Files, lines, parsing and error kinds compose through one io::Result.
/// Read `key=value` lines; every failure path is visible in the signature.
fn load_pairs(path: &Path) -> std::io::Result<Vec<(String, String)>> {
    let text = fs::read_to_string(path)?;
    let mut pairs = Vec::new();
    for line in text.lines() {
        let (k, v) = line.split_once('=').ok_or_else(|| {
            std::io::Error::new(std::io::ErrorKind::InvalidData, format!("bad line: {line:?}"))
        })?;
        pairs.push((k.trim().to_string(), v.trim().to_string()));
    }
    Ok(pairs)
}
fs::create_dir_all("scratch07").unwrap();
fs::write("scratch07/pairs.txt", "host=ny4\nrate=40\n").unwrap();
println!("ok        -> {:?}", load_pairs(Path::new("scratch07/pairs.txt")));
println!("missing   -> {:?}", load_pairs(Path::new("scratch07/nope.cfg")).err().map(|e| e.kind()));
fs::write("scratch07/broken.cfg", "host=ny4\nno equals sign\n").unwrap();
println!("malformed -> {:?}", load_pairs(Path::new("scratch07/broken.cfg")).err().map(|e| e.kind()));


Three different failure origins — file missing, file unreadable, file malformed — arrive through one `io::Result` and separate cleanly by `ErrorKind`. That is the design chapter 11 will formalise into real error types, and it is why nothing in this chapter ever needed a panic.

## 14. Habits and Review

The reflexes this chapter should leave behind:

- **Return `io::Result` from your own functions** and let `?` do the propagating; unwrap only at the edge where the program can do nothing but stop.
- **Match on `ErrorKind` for decisions, `Display` for messages.** Logic must not depend on error *strings*.
- **Take `impl AsRef<Path>` (or `&Path`) in function signatures**; build `PathBuf` at the call site.
- **Wrap loop writes in `BufWriter` and end with `flush()`** — the drop flush is silent on failure.
- **Use append mode for anything that only grows**; use write-temp-then-`rename` for anything you replace.
- **Prefer `args_os()`/`to_string_lossy()` at program boundaries**; treat text conversion as an explicit step, never an accident.
- **Set env vars only inside `unsafe`, in single-threaded setup code** — and prefer *reading* configuration over mutating it.
- **Check `status.success()` before trusting captured output** — a non-zero exit is data, not an error return.

## 15. Cheat Sheet

| Task | Code |
|---|---|
| read whole text | `fs::read_to_string(path)?` |
| read whole bytes | `fs::read(path)?` |
| write/overwrite | `fs::write(path, data)?` (truncates) |
| open with options | `OpenOptions::new().append(true).create(true).open(path)?` |
| line iterator | `BufReader::new(f).lines().map(\|l\| l?)` |
| one line, keep `\n` | `reader.read_line(&mut buf)?` |
| write formatted | `writeln!(handle, "{}", value)?` |
| flush pending writes | `writer.flush()?` |
| list a directory | `fs::read_dir(dir)?.filter_map(\|e\| e.ok())` (+ sort) |
| file size/type | `entry.metadata()?.len()`, `.is_file()`, `.is_dir()` |
| move / rename | `fs::rename(from, to)?` |
| remove | `fs::remove_file(p)?`, `fs::remove_dir_all(d)?` |
| exists? | `fs::exists(p)?` (no error on absence) |
| absolute form | `fs::canonicalize(p)?` (`\\?\` prefix on Windows) |
| arguments | `env::args()`, `.nth(n)` → `Option<String>`; `args_os()` for no-panic |
| read env var | `env::var("KEY")` → match `VarError::NotPresent` |
| set/remove env | `unsafe { env::set_var(k, v) }` (edition 2024) |
| run a program | `Command::new(p).arg(a).output()?` / `.status()?` |
| exit code as data | `out.status.success()`, `status.code()` → `Option<i32>` |

## 16. Review Questions

Answer from memory; each one is a §-numbered cell away.

1. Why is `eprintln!` safe to use in a crash path while a `BufWriter`'s pending data is not? (§2, §6)
2. What does `fs::read_to_string` return when the file contains invalid UTF-8, and which `ErrorKind` is it? (§4)
3. Cell 5.2 opened the same log twice, once per "session". What makes the second `writeln!` land after the first? (§5)
4. You pushed `"logs"` then `"2026"` into a `PathBuf` on Windows. What separates them in the final string, and who decided that? (§3)
5. `Command::new("nope").output()` and `Command::new("cmd").args(["/C", "exit 3"]).output()` — which is `Err`, which is `Ok`, and what does the `Ok` carry? (§10)
6. In edition 2024, what must surround `env::set_var`, and what property of your program makes the call safe? (§9)
7. A single `write!(buf_writer, very_long_bytes)` larger than the buffer lands in the file *before* any flush. Why does that not contradict §6? (§6)
8. What is element 0 of `env::args()`, and what type does `.nth(2)` hand you for a two-argument invocation? (§8)

### What Comes Next

Chapter 07 closes the core-toolbox arc: your programs can now take input, hold state, and touch the world — and name every way the world can disappoint them.

| Chapter | Title | What it settles |
|---|---|---|
| **08** | **Assertions and Type Checks** | `assert!`, `matches!`, `size_of` — checking the assumptions this chapter's code made |
| 09 | Logging | `tracing` events and spans; the structured answer to "what did the program actually do" — including the I/O this chapter only printed |
| 11 | Errors in Depth | real error types for the `io::Error`s you matched by kind here; `?` across error types |

**Carry forward.** You should now be able to do these without looking anything up:

- read and write files three ways (one-shot, handle, buffered) and say when each is right
- classify an `io::Error` by kind, and explain why kinds beat message strings
- build and decompose paths with `Path`/`PathBuf`, knowing what Windows does to separators
- read `args()` and `env::var` without a panic path in sight
- spawn a program, capture its output, and read its exit code as data
- predict where the syscalls are in any I/O loop you write — and buffer them away
